<a href="https://colab.research.google.com/github/DavidSenseman/STA1403/blob/main/Chapter_11B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- STA1403_CHAPTER_11B:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **STA 1403: Biostatistics**

## **Chapter 11 : Regression Analysis**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

#### Contents
* 11.1 : Introduction
* 11.2 : Linear Regression Models with One Binary Explanatory
Variable
* 11.3 : Statistical Inference Using Simple Linear Regression
Models
* 11.4 : Linear Regression Models with One Numerical
Explanatory Variable
* **11.5 : Goodness of Fit**
* **11.6 : Model Assumptions and Diagnostics**
* **11.7 : Multiple Linear Regression**
* **11.8 : Advanced**

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which **_is_** your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.139.115.187",
  "hostname": "187.115.139.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## Test Your STA1403 Key

Run the next cell to test whether your STA1403 Key is correctly installed in your Colab Secrets.

In [ ]:
# @title Test Your STA1403 KEY

from google.colab import userdata
import os

# Check if STA1403 key is properly loaded
try:
    # 1. Get the key from Secrets
    STA1403_KEY = userdata.get('STA1403_KEY')

    # 2. Set it as an environment variable
    os.environ['STA1403_KEY'] = STA1403_KEY

    print("STA1403 key loaded and environment variable set successfully!")
    print(f"Key length: {len(STA1403_KEY)}")

except Exception as e:
    print(f"Error loading STA1403 key: {e}")
    print("Please set your STA1403 key in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'STA1403_KEY'")
    print("3. Paste your STA1403 key and toggle 'Notebook access' on")

If your STA1403 key is correctly stored in your Colab Secrets, you should see the following output:
```text
STA1403 key loaded and environment variable set successfully!
Key length: 28
```
Contact your Instructor if you recieve an error since you won't be able to submit your lesson for grading unless your STA1403 key is correctly installed.

## Load Data Sets

Run the next cell to load the data sets for this lesson.

In [ ]:
# @title Load Data Sets
import pandas as pd

# Load birthwt dataset
birthwt_df = pd.read_csv("https://biologicslab.co/STA1403/data/birthwt",
                         sep = ',', na_values=[" ", "NA", "null", ""])

# Load dataset
bodyTemp_df = pd.read_csv("https://biologicslab.co/STA1403/data/BodyTemperature.txt",
                         sep = ' ', na_values=[" ", "NA", "null", ""])


# Load dataset
bodyfat_df = pd.read_csv("https://biologicslab.co/STA1403/data/bodyfat.csv",
                         sep = ',', na_values=[" ", "NA", "null", ""])

# Load dataset
saltBP_df = pd.read_csv("https://biologicslab.co/STA1403/data/saltBP.txt",
                         sep = ' ', na_values=[" ", "NA", "null", ""])

print("Data sets loaded sucessfully. ✅")

# **Chapter 11 :  Regression Analysis**

## **11.5 Goodness of Fit**

When we fit a least-squares regression model to the observed data, Python provides some other important information about our model besides the Coefficients table. Here, we focus on **R-squared**, which measures how well the regression model fits the observed data. We denote this measure as $ R^{2}$.

Recall that the residual sums of squares (RSS) quantifies the discrepancy between the observed data and the regression line used to represent the data: the higher RSS, the higher discrepancy. Therefore, we can interpret RSS as the **unexplained variation** in the response variable using the regression model.

Alternatively, we can interpret RSS as the **lack of fit** of the linear regression model. In contrast, $ R^{2} $ is a measure of goodness of fit; that is, how well our model represents the observed data and explains the variation in the response variable. The value of $ R^{2} $ is between 0 and 1, and the better the model fits the data, the higher its $R^{2}$ is.

The total variation in the response variable before we fit the regression line is called the **Total Sum of Squares** (TSS) and is calculated as the squared deviations of each observed value of the response variable, $y_{i}$, from its sample mean $\bar{y}$ :
$$
TSS = \sum_{i}^{n} \left(y_{i}- \bar{y}\right)^{2}.
$$
The fraction RSS/TSS can be interpreted as the percent of total variation that was not explained by the regression model.

-------------------

>In contrast, $1- RSS/TSS$ is fraction of total variation explained by the model. This fraction is $ R^{2} $ , which measures the goodness of fit for the regression model,
$$ R ^ {2} = 1 - \frac {\mathrm {RSS}}{\mathrm {TSS}}. $$

------------------


## Example 1: Print Measures of Goodness of Fit

The code in the cell below prints out measures of goodness of fit for a linear regression model using the `BodyTemperature` dataset. The model uses the variable `HeartRate` as the "predictor" with the variable `Temperature` as the dependent variable.

**Code Note:**

The results of the regression analysis have been formated to emulate how the analysis would appear if you were using the $R$ programming language.   

In [ ]:
# @title Example 1: Print Measures of Goodness of Fit

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
df            = bodyTemp_df       # DataFrame
dependent_var = "Temperature"     # Response variable
predictor     = "HeartRate"       # Predictor
# ──────────────────────────────────────────────────────────────────────────────

# 2. Simple linear regression
formula = f"{dependent_var} ~ {predictor}"
model = ols(formula, data=df).fit()

# 3. Extract stats directly from model
rse      = np.sqrt(model.mse_resid)
df_resid = int(model.df_resid)
r2       = model.rsquared
r2_adj   = model.rsquared_adj
f_stat   = model.fvalue
f_df1    = int(model.df_model)
f_df2    = int(model.df_resid)
f_pval   = model.f_pvalue

fmt_p = lambda p: f"{p:.3e}" if p < 0.001 else f"{p:.4f}"

# 4. Print Measures of Goodness of Fit
print(f"Residual standard error: {rse:.3f} on {df_resid} degrees of freedom")
print(f"Multiple R-squared:  {r2:.4f},\tAdjusted R-squared:  {r2_adj:.4f}")
print(f"F-statistic: {f_stat:.2f} on {f_df1} and {f_df2} DF,  p-value: {fmt_p(f_pval)}")
# print(model.summary())

If the code is correct, you should see the following output:
```text
Residual standard error: 0.860 on 98 degrees of freedom
Multiple R-squared:  0.2004,	Adjusted R-squared:  0.1923
F-statistic: 24.56 on 1 and 98 DF,  p-value: 3.011e-06
```

----------------------

**NOTE:** The code in Example 1 has been extensively modified to generate an output that closely resembles the equivalent output when using the $R$ programming language. However, starting with Example 2, basic, unmodified Python will be used. Unless you specifically need to emulate $R$ code, you should use the unmodified Python code later in this lesson for performing linear regression analysis.  

-----------------------

## **Exercise 1: Print Measures of Goodness of Fit**

In the cell below, write the Python code to print out the measures of goodness of fit for the model with `salt` and the explanatory variable for `BP`. For this model, $R^2 = 0.70$.

**Code Hints:**

1. Copy-and-paste Example 1 into the cell below.
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:
```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
df            = saltBP_df   # DataFrame
dependent_var = "BP"        # Response variable
predictor     = "salt"      # Predictor
# ──────────────────────────────────────────────────────────────────────────────
```

In [ ]:
# @title Exercise 1: Print Measures of Goodness of Fit




If the code is correct, you should see the following output:
```text
Residual standard error: 2.745 on 23 degrees of freedom
Multiple R-squared:  0.7036,	Adjusted R-squared:  0.6907
F-statistic: 54.59 on 1 and 23 DF,  p-value: 1.631e-07
```


Focus on the value of $R^{2}$ provided by Python for the linear regression model with blood pressure as the response variable and daily sodium chloride intake as the explanatory variable. For this model, $R^{2} = 0.70$. Therefore, 70% of the total variation in blood pressure can be explained by the daily amount of sodium chloride a person consumes. The remaining $30%$ of the total variation cannot be explained by this model and is regarded as random.

---------------

>For simple linear regression models with one numerical explanatory variable, $ R^{2} $ is equal to the square of the correlation coefficient r.

---------------

For the above blood pressure example, the sample correlation coefficient between blood pressure and daily sodium chloride intake is r = 0.84. Therefore, the $ R^{2} $ is
$$ R ^ {2} = 0. 8 4 ^ {2} = 0. 7 0. $$
This is the same value provided by Python in the output above (**Exercise 1**).

While $R^{2}$ provides useful information about the fit of the regression model, one should be cautious about overstating its importance. Having a large $R^{2}$ only means that the model provides estimates close to the observed values for individuals in our sample. This may or may not translate to better predictions for other individuals that are not included in our sample. Moreover, even when the value of $R^{2}$ is small, the model could still be useful for predicting unknown values of the response variable, especially when we consider the alternative option of not using the model (and the explanatory variable) for prediction.

## **11.6 Model Assumptions and Diagnostics**

Statistical inference using linear regression models is based on several assumptions. Violating these assumptions could lead to wrong conclusions.

_Linearity_ &nbsp;&nbsp;&nbsp;&nbsp; The most important assumption of linear regression model is that the relationship between the explanatory variable $X$ and the response variable $Y$ is linear. For simple problems discussed in this book, we can visually evaluate the appropriateness of this assumption using the scatterplot of $Y$ versus $X$ such as the one shown in Chapter 11A.

When the linearity assumption does not hold, we might still be able to use linear regression models after transforming the original variables. Common transformations are logarithm (usually for the response variable), square root, and square (usually for predictors). For example, we can create a new variable $X^{2} = X_{1}^{2}$ and include it in the regression model to account for possible nonlinear (in this case, parabolic) relationship between the response variable $Y$ and the predictor $X_{1}$.
$$
Y = \alpha + \beta_{1} X_{1} + \beta_{2} X_{1}^{2} + \varepsilon.
$$
We should be cautious about interpreting the model parameters. Note that the role of the quadratic term $X_{1}^{2}$ is to capture possible nonlinearity in the relationship between $Y$ and $X_{1}$. In this case, the null hypothesis $H_{0}:\beta_{2}=0$ indicates that the relationship is not quadratic. Fitting such models is discussed in the next section.

Finding the right transformation is not trivial. Additionally, variable transformations make the interpretation of the results difficult. Therefore, these techniques are usually more appropriate when our objective is predicting the unknown values of the response variables as opposed to explaining its relationship with a set of explanatory variables. For more discussion, refer to Harrell (2001) [10].

_Independence_ &nbsp;&nbsp;&nbsp;&nbsp; Another important assumption is that the observations are **independent**, which is a reasonable assumption if we use simple random sampling to select individuals that are not related to each other and if we do not obtain multiple observations from the same individual. However, we sometimes sample related subjects (e.g., siblings) in groups. Also, we sometimes select unrelated subjects, but obtain multiple measurements (e.g., over a period of time) of the response variable for each subject. For example, when evaluating the effect of different diets on blood pressure, we might obtain multiple measurements of blood pressure for each person repeatedly over six months. For such data, we need to use regression models that take the dependencies among observations into account. When multiple observations are obtained over time, we typically use a class of statistical models called **longitudinal models**.

_Constant Variance and Normality_ &nbsp;&nbsp;&nbsp;&nbsp;  Using linear regression models also involves some assumptions regarding the probability distribution of the response variable $Y$, which is the main random variable in regression analysis. However, because of the connection between the response variable and the error term, $\varepsilon$, according to the linear regression model (Eq. 11.2), it is common to treat $\varepsilon$ as a random variable (its values change from one individual to another) and specify these assumptions in terms of the probability distribution of $\varepsilon$.

$$\hat{y} = a + b x. \tag{11.2} $$

To make the statistical inference methods we discussed earlier in this chapter valid, it is common to assume that the error term is normally distributed with mean 0,

$$\varepsilon \sim N \left(0, \sigma^ {2}\right).$$

Minor deviations from normality will not have a substantial impact on the results as long as the sample size is relatively large. The important aspects of this assumption are its specifications of the population mean and variance of $\varepsilon$. The population mean is assumed to be zero, so we expect the errors based on the regression line for the whole population to be centered on zero. The population variance of $\varepsilon$ is $ \sigma^{2}$, which is of course unknown. What we actually mean by this assumption is that whatever the value of this parameter, it does not change for different values of the explanatory variable, e.g., $\sigma^{2}$ remains the same for $x = 5$ and $x = 10$. Informally, this means that we expect that the variation of the actual values of the response variable around the regression line remains the same regardless of the value of the explanatory variable. This is called the **constant variance** assumption, which is also known as the **homoscedasticity** assumption. (Heteroscedasticity refers to situations where the constant variance assumption is violated.) To check the validity of these assumptions, we examine the residuals e that are observed values of the random variable $\varepsilon$.


To illustrate how we check the assumptions regarding $\varepsilon$, we use the blood pressure example with daily amount of sodium chloride intake as the explanatory variable.

![__](https://biologicslab.co/STA1403/images/chapter_11/chapter_11B_image01A.png)

>**Fig. 11.11** _Left panel:_ The residual plot for the blood pressure example (with daily amount of sodium chloride intake as the explanatory variable) to assess the assumptions of linear regression models related to the error term. Here, the scatter plot of residuals vs. the estimated (fitted) values is shown. The _horizontal axis_ represents the regression line. The _solid line_ on the plot shows the overall pattern of the residuals. The plot shows that the residuals are scattered randomly around the _horizontal dashed line_ at zero without any detectable pattern. _Right panel:_ An illustrative example, where the constant variance assumption is violated. Here, the residuals become more dispersed around the _horizontal line_ as we move from small to large fitted values.

The residual plot in the left panel of Fig. 11.11 shows the residuals $ e $ versus the estimated (fitted) values of the response variable $ \hat{y} $ . The horizontal line represents the regression line. The plot shows that the residuals are scattered randomly around the horizontal dashed line at zero without any detectable pattern. In this figure, the solid line shows the overall pattern of the residuals. This line should remain close to the horizontal dashed line. Moreover, it is important that we do not see any non-random pattern in the residual plot. For example, we should not see small variations around the horizontal line in one region and high variations in another region. For illustration purposes, the right panel of Fig. 11.11 shows a residual plot where the variability of the residuals around the horizontal line changes from one region to another. More specifically, for this example, residuals become more dispersed around the horizontal line as we move from small to large fitted values. In this case, the constant variance assumption is violated.

When the constant variance assumption does not hold, we can sometimes stabilize the variance using simple transformations of the response variable so the variance becomes approximately constant. For example, instead of $Y$, we could use $\sqrt{Y}$ (usually when $Y$ is a count variable), or $\log(Y)$ in the regression model. Another strategy is to use _weighted least squares_ (not discussed in this book) instead of the standard least squares approach.

In Fig. 11.11, the observations with large residuals (in absolute value) are identified by their row numbers. For these observations, the relationship between the response variable and the explanatory variable does not follow the overall pattern closely. We usually regard such observations as outliers and investigate them further to make sure that they are measured and recorded correctly. Again, we should not remove outliers from the data unless we are absolutely sure that they are recorded by mistake. (Occasionally, we remove outliers temporarily and refit the model to examine the extent of their influence on the regression model.)

------------------

>We use the residuals as the observed values of the error terms, $ \varepsilon $ , to estimate its unknown population standard deviation $ \sigma $ as follows:
$$SE_{e} = \sqrt {RSS / (n - 2)}.$$
We refer to $ SE_{e} $ as the **regression standard error**. This is in fact the numerator in Eq. 11.3 for the standard error of the regression coefficient. Therefore, we can rewrite Eq. 11.3 as follows:
$$
SE_{b} = \frac {SE_{e}}{\sqrt {\sum_{i} \left(x_{i}-\bar{x}\right)^{2}}}. \tag{11.5}
$$

The regression standard error for the blood pressure example with the daily sodium chloride intake as the explanatory variable is $ SE_{e}=2.745 $. This value is shown in the output of **Exercise 1**. Divide this value by $\sqrt{\sum_{i}(x_{i}-\bar{x})^{2}}$ to see that the result is equal to $0.162$, the standard error of the regression coefficient.

## **11.7 Multiple Linear Regression**

So far, we have focused on linear regression models with only one explanatory variable. In most cases, however, we are interested in the relationship between the response variable and multiple explanatory variables. Even if we are interested in the relationship between the response variable and only one explanatory variable, very often we need to account for the effect of other important variables that might influence our inference. If our objective is to predict unknown values of the response variable, we might be able to improve prediction accuracy by including multiple predictors in the linear regression model. Models with multiple explanatory variables or predictors are called **multiple linear regression** models.

As an example, suppose that we want to examine the relationship between the birthweight of babies and the smoking status of their mothers during pregnancy. We might however believe that mother's age at the time of pregnancy is an important factor that should be taken into account. To this end, we need to evaluate the relationship between birthweight and smoking status among mothers with similar age.

Alternatively, suppose that our objective is to predict birthweight given age and smoking status of mothers; that is, we are interested in predicting birthweight if, for example, we are told that the mother has been smoking during pregnancy and she is 30 years old. Again, we need to include both age and smoking status in our model.

--------------

>In practice, we need to specify our objective for building linear regression models clearly. If our objective is to examine possible relationships between the response variable and one or more explanatory variables, we should specify our hypothesis prior to our analysis. In this case, our decision to include an explanatory variable in the model must be hypothesis-driven and based on our domain knowledge. When testing a hypothesis, we should avoid finding our model through exploring all possible combinations of variables available to us. On the other hand, if our objective is to predict the unknown values of the response variable, we could use our domain knowledge to identify a set of promising predictors and then explore all possible combinations of these variables until we find a model that provides the best predictions.

--------------

Note that our criterion for finding the best regression model for prediction should be based on how the model predicts unknown values of the response variable (e.g.,for the part of the population not included in our sample). To this end, a common criterion is the **mean squared error** (MSE), which measures how close the predicted values are to the actual values,

$$
\mathrm{MSE} = \frac{1}{N} \sum_{i = 1}^{N} \left(y _{i}-\hat{y}_{i}\right)^{2}.
$$

When used for evaluating predictions, MSE is sometimes called the Expected Prediction Error (EPE)[11] or the Mean Squared Error of Prediction (MSEP)[3]. Note that the sum is over all unknown values (i.e., the whole population). In practice, we cannot calculate MSE directly since the actual values of the response variable are unknown. We can however estimate it using a subset of our sample as the **test set**. These are observations we remove from our sample before fitting the regression model and treat them as future observations. That is, we pretend that we do not know the value of the response variable for these observations. We then use the remaining observations, called the **training set**, in our sample to fit the regression model and use the resulting model to predict the response variable for the test set. Suppose we have m observations in the test set, we estimate MSE as follows:
$$
\widehat {\mathrm {MSE}} = \frac {1}{m} \sum_ {i = 1} ^ {m} \left(y _ {i} - \hat {y} _ {i}\right) ^ {2}.
$$
Here, $ \hat{y}_{i} $ is our prediction for the $i$ th observation in the test set, and $ y_{i} $ is the actual value of the response variable for this observation. When we use regression models for prediction, we prefer models with small $\widehat {\mathrm {MSE}}$.

While fitting a multiple linear regression model to the data follows the same principle (namely, minimizing the residual sum of squares) as what we used for simple linear regression model, estimating regression parameters and performing statistical inference using multiple linear regression models is slightly more complex than what we discussed for simple linear regression models. Here, we focus on using Python to find parameter estimates and interpret the results.

A multiple linear regression model with $p$ explanatory variables can be presented as follows:
$$
Y = \alpha + \beta_{1} X_{1} + \beta_{2} X_{2} + \dots + \beta_{p} X_{p} + \varepsilon .
$$

Here, $Y$ is the numerical response variable, $X_{1}, \ldots, X_{p}$ are the explanatory variables, $\alpha$ is the intercept, $\beta_{1}, \ldots, \beta_{p} $ are the corresponding regression coefficients, and $ \varepsilon $ is the error term.

Using the observed data, we estimate the regression parameters $ \alpha, \beta_{1}, \ldots, \beta_{p} $ . To this end, we use the least-squares method as before. We denote the point estimates for these parameters $ a, b_{1}, \ldots, b_{p} $ , respectively. Using the point estimates, we can predict the value of the response variable for an individual whose measurements for the explanatory variables are $ x_{1}, \ldots, x_{p}$ :

$$
\hat {y} = a + b_{1} x_{1} + b_{2} x_{2} + \dots + b_{p} x_{p}.
$$

For the $i$th observed data, the difference between the actual value of the response variable, $y_{i}$, and its estimate according to the above linear regression model, $\hat{y}_{i}$, is the residual $e{i}$,
$$
e_{i} = y_{i} - \hat {y}_{i}.
$$
As before, we measure the discrepancy between the model and the observed data using $RSS$, which is the sum of the square of the residuals. As before, $R^{2}$ measures the goodness of fit for the regression model,
$$
R^{2} = 1 - \frac {RSS}{TSS}.
$$

Using $RSS$, we can find the regression standard error (i.e., the estimate of $\sigma$) as follows:
$$
SE _{e} = \sqrt {RSS / (n-p-1)}.
$$
where $n$ is the sample size, and $p$ is the number of explanatory variables in the model.
For each regression coefficient $ \beta_{j} $ (i.e., the coefficient of the explanatory variable $ X_{j}$), we find the standard error $SE_{b_{j}}$ along with its point estimate $b_{j}$. Similar to simple linear regression models, confidence intervals for $ \beta_{j} $ are obtained as follows:
$$
[b_{j}-t_{\mathrm {crit}} \times SE_{b_{j}}, b_{j} + t_{\mathrm {crit}} \times SE_{b_{j}}].
$$
Here, however, we obtain $ t_{crit} $ from the $t$-distribution with $n - p - 1$ degrees of freedom for the given confidence level $c$.

The steps for performing hypothesis testing regarding the $\beta_{j}$ is also similar to what we discussed for simple linear relationship models; the null hypothesis is $H_{0}: \beta_{j}=0 $; we evaluate the null hypothesis by finding the $t$-score,
$$
t_{j} = \frac{b_{j}}{SE_{b _ {j}}}.
$$
However, for multiple linear regression models, we obtain $p$-values (i.e., the observed significance level) using the $t$-distribution with $n - p - 1$ degrees of freedom.

## Example 2: Multiple Linear Regression

The code in the cell below demonstrates how to perform multiple linear regression using the `birthwt` dataset. The linear regression model exams the hypothesis that both the variables `age` and `smoke` are predictors of `bwt` (body weight).

In [ ]:
# @title Example 2: Multiple Linear Regression

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
df               = birthwt_df   # DataFrame
dependent_var    = "bwt"        # Response variable
predictor_1      = "age"        # Predictor 1
predictor_2      =  "smoke"     # Predictor 2
interaction_term = "+"       # Interaction term
# ──────────────────────────────────────────────────────────────────────────────

# Multiple linear regression
formula = f"{dependent_var} ~ {predictor_1} {interaction_term} {predictor_2}"
model = ols(formula, data=df).fit()
print(f"Formula for regression model: {formula}\n")

# Print model summary ───────────────────────────────────────────────
print(model.summary())

If the code is correct, you should see the following output:
```text
Formula for regression model: bwt ~ age + smoke

                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    bwt   R-squared:                       0.042
Model:                            OLS   Adj. R-squared:                  0.032
Method:                 Least Squares   F-statistic:                     4.119
Date:                Sun, 14 Jun 2026   Prob (F-statistic):             0.0178
Time:                        13:05:29   Log-Likelihood:                -1509.4
No. Observations:                 189   AIC:                             3025.
Df Residuals:                     186   BIC:                             3035.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   2791.8214    240.958     11.586      0.000    2316.459    3267.183
age           11.2326      9.882      1.137      0.257      -8.262      30.727
smoke       -276.3201    106.991     -2.583      0.011    -487.391     -65.249
==============================================================================
Omnibus:                        4.111   Durbin-Watson:                   0.228
Prob(Omnibus):                  0.128   Jarque-Bera (JB):                3.956
Skew:                          -0.354   Prob(JB):                        0.138
Kurtosis:                       3.014   Cond. No.                         111.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.

```

As you can see, the summary format contains a large number of statistics related to the analysis of our linear model. For now, just focus on this part of the output
```text
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   2791.8214    240.958     11.586      0.000    2316.459    3267.183
age           11.2326      9.882      1.137      0.257      -8.262      30.727
smoke       -276.3201    106.991     -2.583      0.011    -487.391     -65.249
```
which provides the intercept and the regression coefficients of mother's age and her smoking status (`smoke`). Using these point estimates for regression parameters, we can predict birthweight for a baby by knowing her mother's age and smoking status as follows:

$$
\widehat {bwt} = 2792 + 11 \times age - 276 \times smoke.
$$

Therefore, if the mother is 30 years old (`age=30`) and she has been smoking during the pregnancy (`smoke=1`), our estimate (`prediction`) for the birthweight of her baby is
$$
\begin{array}{l} \widehat {bwt} = 2791 + 11 \times 30 - 276 \times 1 \\
= 2845. \\
\end{array}
$$
This can be interpreted as our estimate of the expected birthweight for babies whose mothers are $30$ years old and smoke during pregnancy. That is, for these mothers, the expected (average) birthweight of babies is $2845$ grams.

The intercept in multiple linear regression model is the expected (average) value of the response variable when all the explanatory variables in the model are set to zero simultaneously. In the above example, the intercept is $a=2791$, which is obtained by setting age and smoking to zero. We might be tempted to interpret this as the average birthweight of babies for nonsmoking mothers (smoke=0) with age equal to zero. In this case, however, this is not a reasonable interpretation since mother's age cannot be zero.

------------

>For multiple linear regression models, we use $b_{j}$ to denote the point estimate of the regression coefficient $\beta_{j}$. We interpret $b_{j}$ as our estimate of the expected (average) change in the response variable associated with a unit increase in the corresponding explanatory variable $X_{j}$ while all other explanatory variables in the model remain fixed.

-------------

For the above birthweight example, the point estimate of the regression coefficient for age is $b_{1}=11$ (output of Example 2). Therefore, we expect that the birthweight of babies increase by 11 grams as the mother's age increases by one year among mothers with the same smoking status. If we select two groups of mothers with the same smoking status from the population where the first group includes mothers who are, for example, 27 years old, and the second group includes mothers who are 28 years old, the average birthweight for the second group is $11$ grams higher than the average birthweight in the first group according to our model.

For this model, the estimate of the regression coefficient for smoke is $b_{2}=-276$. Therefore, the expected birthweight decreases by $-276$ grams associated with one unit increase in the value of the variable smoke among mothers with the same age. In this case, because smoke is a binary variable, one unit increase in its value means changing the smoking status from nonsmoking (`smoke=0`) to smoking (`smoke=1`). Note that the age of mothers should remain fixed to make this interpretation valid. For example, if we divide mothers who are $32$ years old into two groups according to their smoking status, the expected weight of birthweight for smoking mothers is $276$ grams less than that of nonsmoking mothers.

![__](https://biologicslab.co/STA1403/images/chapter_11/chapter_11B_image02A.png)

>**Fig. 11.14** Presenting the multiple linear regression model fitted to the birthwt data. Here, nonsmoking mothers are shown as circles, while smoking mothers are shown as squares. The dashed line shows the regression line among nonsmoking mothers, and the solid line shows the regression line among the smoking mothers.


Figure 11.14 shows the above multiple linear regression model, where the linear relationship between birthweight and age is captured by a separate regression line for each smoking status. In this plot, nonsmoking mothers are shown as circles, while smoking mothers are shown as squares. The dashed line shows the regression line among nonsmoking mothers, and the solid line shows the regression line among the smoking mothers. Note that the slopes of the two lines are the same and are equal to $11$. Therefore, the expected change in birthweight associated with one unit increase in age remains the same regardless of smoking status. On the other hand, the vertical distance between the two lines remains equal to $276$ over all possible values of age. Therefore, for any given age, we expect the same difference in birthweight between children of smoking and nonsmoking mothers.

------------

>In multiple linear regression models, we usually assume that the effects of explanatory variables on the response variable are **additive**. This means that the expected change in the response variable corresponding to one unit increase in one of the explanatory variables remains the same regardless of the values of other explanatory variables in the model. As a result, when two (or more) explanatory variables change simultaneously, their overall effect on the response variable is the sum of their individual effects. For example, if we change mother's age from $27$ to $28$ and change smoking status from $0$ to $1$, the expected value of birthweight (in grams) changes by $11 + (-276) = -269$.

The coefficient table in the output from **Example 2** provides the standard errors along with the point estimate of the each regression coefficient. For this example, the standard errors are roughly $SE_{b_{1}} = 10$ and $SE_{b_{2}}= 107$. Suppose that we are interested in the 95% confidence intervals for $\beta_{1} $ and $ \beta_{2}$. To this end, we first need to find $t_{crit} $ for 0.95 confidence level from the $t$-distribution with $df = 189 - 2 - 1 = 186$. (Note that the sample size is $n = 189$ and the number of explanatory variables is $p = 2$.)

Using Python, we find $t_{crit} = 1.97$ for 0.95 confidence level. (The process of finding $t_{crit}$ is the same as what we discussed for the population mean in Chap. 6.) Therefore, the 95% confidence interval for regression coefficient of age is
$$
[11 -1.97 \times 10, 11 + 1.97 \times 10 ] = [- 8,31].
$$

Likewise, the $95$% confidence interval for the regression coefficient of smoke is
$$
[-276 - 1.97 \times 107, -276 + 1.98 \times 107] = [-489, -67].
$$


According to the above results, at $0.95$ confidence level, our expected change in birthweight associated with one year increase in mother's age among mothers with the same smoking status is somewhere between $-8$ and $31$ grams. Note that this range includes zero, which is the value specified by the null hypothesis. More formally, considering the $p$-value of $0.25$ provided in the output for **Example 2** for the regression coefficient for age, we cannot reject its corresponding null hypothesis at commonly used significant levels ($0.01$, $0.05$, and $0.1$). The null hypothesis in this case states that birthweight and age are not linearly related ($\beta_{1}=0$). We can, however, reject the null hypothesis for the regression coefficient for the smoking status at $0.05$ level and conclude that the result of this test is statistically significant.

For the above example, suppose we remove the variable age from the model and fit a new linear regression model with the variable `smoke` only. By doing so, our inference regarding the linear relationship between birthweight and smoke changes. This will be demonstrated by **Exercise 2**.

## **Exercise 2: Multiple Linear Regression**

Repeat Example 2, but remove the variable `age` from the model.

**Code Hints:**

1. Copy-and-paste Example 2 into the cell below.
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
df                = birthwt_df   # DataFrame
dependent_var     = "bwt"        # Response variable
predictor_1       = ""           # Predictor 1
predictor_2       =  "smoke"     # Predictor 2
interaction_term = "+"           # Interaction term
# ──────────────────────────────────────────────────────────────────────────────
```

In [ ]:
# @title Exercise 2: Multiple Linear Regression



If the code is correct, you should see the following output:

```text
Formula for regression model: bwt ~  + smoke

                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    bwt   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                  0.031
Method:                 Least Squares   F-statistic:                     6.936
Date:                Sun, 14 Jun 2026   Prob (F-statistic):            0.00916
Time:                        20:34:48   Log-Likelihood:                -1510.1
No. Observations:                 189   AIC:                             3024.
Df Residuals:                     187   BIC:                             3031.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   3054.9565     66.933     45.642      0.000    2922.915    3186.998
smoke       -281.7133    106.969     -2.634      0.009    -492.734     -70.693
==============================================================================
Omnibus:                        2.742   Durbin-Watson:                   0.234
Prob(Omnibus):                  0.254   Jarque-Bera (JB):                2.676
Skew:                          -0.290   Prob(JB):                        0.262
Kurtosis:                       2.936   Cond. No.                         2.44
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
```

In this case, by including smoke as the only explanatory variable, the estimate of the regression coefficient for this variable changes from $-276$ to $-281.7$, and the $p$-value changes from $0.010$ to $0.009$.

For the above example, including and removing age did not have a substantial impact on our inference regarding the relationship between birthweight and smoking status. In some situations, the impact of adding or removing an explanatory variable could be drastic. As an example, suppose we want to examine the relationship between height and percent body fat among men. To do this, we use the bodyfat data set (discussed in Chap. 3), which consists of measurements of percent body fat, age in years, weight in pounds, height in inches, and abdomen circumference in inches for $252$ men. (Follow the steps discussed in Chap. 3 to load the data in Python.)

We build two linear regression models. In the first model, we only include height as the explanatory variable. In the second model, we include height and abdomen both. The steps for fitting linear regression models are the same as before.

## Example 3: Fitting Linear Models

The code in the cell below builds a linear model in which the variable `height_in` is the only explanatory variable. This is the first of two models.

In [ ]:
# @title Example 3: Fitting Linear Models

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
df            = bodyfat_df               # DataFrame
dependent_var = "percent_body_fat"       # Response variable
predictor_1   = ""                       # Predictor 1
predictor_2   = "height_in"              # Predictor 2
interaction_term = "+"                   # Interaction term
# ──────────────────────────────────────────────────────────────────────────────

# Multiple linear regression
formula = f"{dependent_var} ~ {predictor_1} {interaction_term} {predictor_2}"
model = ols(formula, data=df).fit()
print(f"Formula for regression model: {formula}\n")

# Print model summary
print(model.summary())

If the code is correct, you should see the following output:
```text
Formula for regression model: percent_body_fat ~  + height_in

                            OLS Regression Results                            
==============================================================================
Dep. Variable:       percent_body_fat   R-squared:                       0.008
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     2.019
Date:                Sun, 14 Jun 2026   Prob (F-statistic):              0.157
Time:                        20:36:58   Log-Likelihood:                -891.43
No. Observations:                 252   AIC:                             1787.
Df Residuals:                     250   BIC:                             1794.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     33.4945     10.110      3.313      0.001      13.584      53.405
height_in     -0.2045      0.144     -1.421      0.157      -0.488       0.079
==============================================================================
Omnibus:                        2.343   Durbin-Watson:                   1.474
Prob(Omnibus):                  0.310   Jarque-Bera (JB):                1.995
Skew:                           0.099   Prob(JB):                        0.369
Kurtosis:                       2.611   Cond. No.                     1.35e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.35e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
```

The output shows the results for the first model, where we only include `height_in` as the explanatory variable in the model. As we can see, the $p$-value for testing the hypothesis regarding the linear relationship between percent body fat and height is $0.157$, so the result is not statistically significant at $0.1$ level.

## **Exercise 3: Fitting Linear Models**

Repeat the analysis shown in Example 3, but also include the variable `abdomen_cm` as `predictor_1`.

**Code Hints:**

1. Copy-and-paste Example 3 into the cell below.
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
df            = bodyfat_df               # DataFrame
dependent_var = "percent_body_fat"       # Response variable
predictor_1   = "abdomen_cm"             # Predictor 1
predictor_2   = "height_in"              # Predictor 2
interaction_term = "+"                   # Interaction term
# ──────────────────────────────────────────────────────────────────────────────
```

In [ ]:
# @title Exercise 3: Fitting Linear Models



If the code is correct, you should see the following output:
```text
Formula for regression model: percent_body_fat ~ abdomen_cm + height_in

                            OLS Regression Results                            
==============================================================================
Dep. Variable:       percent_body_fat   R-squared:                       0.688
Model:                            OLS   Adj. R-squared:                  0.685
Method:                 Least Squares   F-statistic:                     274.2
Date:                Sun, 14 Jun 2026   Prob (F-statistic):           1.15e-63
Time:                        20:39:22   Log-Likelihood:                -745.78
No. Observations:                 252   AIC:                             1498.
Df Residuals:                     249   BIC:                             1508.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -14.3107      6.043     -2.368      0.019     -26.212      -2.410
abdomen_cm     0.6424      0.028     23.283      0.000       0.588       0.697
height_in     -0.3705      0.081     -4.562      0.000      -0.530      -0.211
==============================================================================
Omnibus:                        1.639   Durbin-Watson:                   1.764
Prob(Omnibus):                  0.441   Jarque-Bera (JB):                1.383
Skew:                          -0.172   Prob(JB):                        0.501
Kurtosis:                       3.117   Cond. No.                     2.38e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.38e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
```

 However, when we include abdomen along with height in the model (output above), the linear relationship between percent body fat and height becomes statistically significant at any commonly used significance level; the $p$-value in this case reduces to an extremely small number $(7.95 \times 10^{-6})$, and the estimate of the regression coefficient changes from $-0.20$ to $-0.37$.

The above results show that the importance and significance of one explanatory variable can be affected by presence or absence of other explanatory variables in the model. In this example, while height by its own does not have a statistically significant linear relationship with percent body fat, the relationship becomes quite significant among men with the same abdomen circumference. (Recall that when interpreting regression coefficient of one explanatory variable in multiple linear regression, we assume that all other explanatory variables are fixed at some specific values.)

--------------

>Our inference regarding the relationship between the response variable and an explanatory variable could depend on what other variables we include in the model. Therefore, we should choose the set of explanatory variables very carefully when we perform statistical inference using multiple linear regression models.

------------

## **11.8 Advanced**

In this section, we discuss linear regression models with interaction terms. We also discuss some commonly used Python functions for linear regression analysis.


### **_11.8.1 Interaction_**

In the previous section, we mentioned that the usual assumption in multiple linear regression models is that the effects are additive. If we believe that the effects are not additive (i.e., the effect of one explanatory variable $ X_{1} $ on the response variable depends on the value of another explanatory variable $ X_{2} $ in the model), we can still use linear regression models by including a new variable $ X_{3} = X_{1}X_{2} $ .
$$ Y = \alpha + \beta_ {1} X _ {1} + \beta_ {2} X _ {2} + \beta_ {1 2} X _ {1} X _ {2} + \varepsilon . $$

The term $X_{1}X_{2}$ is called the **interaction term**. We refer to $\beta_{1}$ and $\beta_{2}$ as the **main effects**, and refer to $\beta_{12}$ as the **interaction effect**.

As before, we use the least-squares method to estimate the model parameters,
$$
\hat{y} = a + b_{1} x_{1} + b_{2} x_{2} + b_{12} x_{1}x_{2}.
$$
When we include an interaction term in our model, we should be cautious about how we interpret model parameters. For simplicity, we assume that $ X_{2} $ in the above model is binary so the value of $ x_{2} $ can be either 0 or 1. When $ x_{2}=0 $ , our estimate of the response variable is as follows:
$$
\hat {y} = a + b _ {1} x _ {1}.
$$
The slope of the least-squares regression line is $ b_{1} $ . Therefore, our estimate of the response variable changes by $ b_{1} $ units for one unit increase in $ x_{1} $ , when we fix $ x_{2} $ at zero.

On the other hand, when $ x_{2}=1 $ , our estimate of the response variable is
$$
\hat{y} = a + b_{1} x_{1} + b_{2} + b_{12} x_{1}.
$$
We can rewrite the above equation as follows:
$$
\hat{y} = (a + b_{2}) + (b_{1} + b_{12}) x_{1}.
$$
In this case, the slope of the least-squares regression line is $b_{1} + b_{12}$, which means that our estimate of the response variable changes by $b_{1} + b_{12}$ units for one unit increase in $x_{1}$, _when we fix $x_{2}$ at one_. Notice that the effect of $X_{1}$ on the response variable Y depends on the value of $X_{2}$.

As an example, we fit a multiple linear regression model to the birthweight data with `bwt` as the response variable. As before, we include both age and smoke as explanatory variables, but this time we include their interaction effect in the model,
$$
\mathrm {bwt} = \alpha + \beta_{1} \mathrm {age} + \beta_ {2} \mathrm {smoke} + \beta_ {12} \mathrm {age} \times \mathrm {smoke} + \varepsilon.
$$

## **Exercise 4: Multiple Linear Regression with Interaction**

The steps for using Python to fit multiple linear regression models with an interaction term is exactly the same as demonstrated in Example 3, but this time enter `bwt ~ age * smoke` under Model Formula. When you use the asterisk symbol “*” instead of the plus symbol “+” to separate the explanatory variables. Python includes the main effects automatically along with the interaction effect in the model.

**Code Hints:**

1. Copy-and-paste Example 3 into the cell below
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
df            = birthwt_df    # DataFrame
dependent_var = "bwt"         # Response variable
predictor_1   = "age"         # Predictor 1
predictor_2   =  "smoke"      # Predictor 2
interaction_term = "*"        # Interaction term
# ──────────────────────────────────────────────────────────────────────────────
```

In [ ]:
# @title Exercise 4: Multiple Linear Regression with Interaction



If the code is correct, you should see the following output:
```text
Formula for regression model: bwt ~ age * smoke

                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    bwt   R-squared:                       0.068
Model:                            OLS   Adj. R-squared:                  0.053
Method:                 Least Squares   F-statistic:                     4.521
Date:                Sun, 14 Jun 2026   Prob (F-statistic):            0.00438
Time:                        20:42:17   Log-Likelihood:                -1506.8
No. Observations:                 189   AIC:                             3022.
Df Residuals:                     185   BIC:                             3035.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   2408.3830    292.237      8.241      0.000    1831.838    2984.928
age           27.6006     12.151      2.271      0.024       3.628      51.573
smoke        795.3815    484.420      1.642      0.102    -160.316    1751.079
age:smoke    -46.3630     20.450     -2.267      0.025     -86.709      -6.017
==============================================================================
Omnibus:                        5.005   Durbin-Watson:                   0.230
Prob(Omnibus):                  0.082   Jarque-Bera (JB):                5.050
Skew:                          -0.397   Prob(JB):                       0.0801
Kurtosis:                       2.900   Cond. No.                         263.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
```

In the output above, note that the interaction term is shown as **age:smoke**. For this model, the point estimates of model parameters are $b_{0}=2408$ , $b_{1}=28$ , $b_{2}=795$ , and $b_{12}=-46$. Therefore, we use the following equation to estimate birthweight:

$$
\widehat {\mathrm {bwt}} = 2408 + 28 \times \mathrm {age} + 795 \times \mathrm {smoke} - 46 \times \mathrm {age} \times \mathrm {smoke}.
$$

Because `smoke` is a binary variable, the interpretation of model parameters are similar to what we discussed above. When `smoke = 0` (i.e., for nonsmoking mothers), the estimate of birthweight changes by $b_{1} = 28$ grams for one year increase in mother's age. When smoke = 1 (i.e., for smoking mothers), the estimate of birthweight changes by $b_{1} + b_{12} = 28-47 = -19$ for one year increase in mother's age. That is, for smoking mothers, the estimate of birthweight decreases as mother's age increases. This is illustrated in Fig. 11.19. In this plot, nonsmoking mothers are shown as circles and smoking mothers are shown as squares. The dashed line shows the regression line for nonsmoking mothers, and the solid line shows the regression line for the smoking mothers. While the slope is positive among non-smoking mothers, it becomes negative among smoking mothers. Compare this plot to Fig. 11.14, which we obtained for the additive (no interaction) model.

For the above model, although the estimate of the main effect for smoke is $796$, we _cannot_ interpret this as our estimate of the expected increase in birthweight for smoking mothers (`smoke = 1`) compared to nonsmoking mothers (`smoke = 0`) regardless of mother's age. The interpretation of smoking effect is slightly complex because it depends on mother's age, which can take many different values. Let us focus on mothers who are 23 years old at the time of pregnancy. This is the average age of mothers in our data. For nonsmoking mothers who are $23$ years old, our estimate of birthweight (in grams) is
$$
\begin{array}{l}
\widehat {\mathrm {bwt}} = 2408 + 28 \times 23 + 795 \times 0 - 46 \times 23 \times 0 \\
= 3050. \\
\end{array}
$$

![__](https://biologicslab.co/STA1403/images/chapter_11/chapter_11B_image03A.png)

>**Fig. 11.19** Presenting the multiple linear regression model with an interaction term fitted to the `birthwt` data. Here, nonsmoking mothers are shown as _circles_, while smoking mothers are shown as _squares_. The _dashed line_ shows the regression line among nonsmoking mothers, and the _solid line_ shows the regression line among the smoking mothers

In contrast, our estimate of birthweight (in grams) for smoking mothers at the same age is
$$
\begin{array}{l}
\widehat {\mathrm {bwt}} = 2406 + 28 \times 23 + 798 \times 1-47 \times 23 \times 1 \\
= 2767. \\
\end{array}
$$
Therefore, the estimated birthweight reduces by $3050 - 2767 = 283$ grams for $23$-year old smoking mothers compared to $23$-year old nonsmoking mothers. Note that this estimate changes depending on mother's age.

# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) Exercises.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/STA1403/backend_code/validate.py?v=" + str(time.time())

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)

with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))

main()